# Label QA Review: Per-Class Galleries + Auto-Flags (pre-training gate)

Reviews every label source **before** any YOLO training:

1. **Per-class statistics** — counts, box sizes, frames without boxes.
2. **Auto-flags** — labels that are probably wrong, found mechanically:
   exploded boxes (tracker grabbed the background), collapsed/tiny boxes,
   extreme aspect ratios, and tracking jumps (box teleports between frames).
3. **Sampled galleries per class** — random frames with boxes drawn, plus a separate
   red gallery of flagged frames so your eyes go where the problems are.
4. **Sequence filmstrips** — all frames of the worst sequences in one strip; the best
   way to see tracker drift.

Works on the click-propagated labels now; add the batch SAM3 label CSVs to
`LABEL_SOURCES` later and rerun — same QA for every source.

**Fix loop:** flagged click-label sequences -> delete their `clicks.json` entries (or use
`REDO_OBJECTS`) in `click_assisted_labeling.ipynb`, re-click, re-propagate, rerun this.

## 1. Setup + config

In [ ]:
import json
import random
import re
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def find_repo_root() -> Path | None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists():
            return base
    colab_clone = Path("/content/vision-uss-research")
    return colab_clone if colab_clone.exists() else None

PROFILE = "colab_drive" if IN_COLAB else "local"
OUTPUTS_ROOT_OVERRIDE = None

repo_root = find_repo_root()
if repo_root:
    sys.path.insert(0, str(repo_root / "src"))
    from vision_uss_research.settings import load_paths
    paths = load_paths(profile=PROFILE)
    OUTPUTS_ROOT = Path(OUTPUTS_ROOT_OVERRIDE or paths.outputs_dir)
else:
    OUTPUTS_ROOT = Path(OUTPUTS_ROOT_OVERRIDE)

# Each source: a boxes CSV + the frames root used to re-resolve stale frame paths.
LABEL_SOURCES = [
    {"name": "click_propagated",
     "boxes_csv": OUTPUTS_ROOT / "experiments" / "click_labels" / "propagated_boxes.csv",
     "frames_root": OUTPUTS_ROOT / "experiments" / "click_labels" / "frames"},
    # add the batch SAM3 text-prompt labels here when they exist:
    # {"name": "sam3_text", "boxes_csv": ..., "frames_root": ...},
]

QA_DIR = OUTPUTS_ROOT / "experiments" / "label_qa"
QA_DIR.mkdir(parents=True, exist_ok=True)

N_GALLERY_PER_CLASS = 12
SEED = 0

# anomaly thresholds (fractions of image area unless stated)
BIG_BOX_AREA = 0.45        # box covers half the image -> tracker exploded
# tiny-box threshold is per class: pole (median area ~0.0013) and woodenboard
# (~0.0006) are GENUINELY tiny - a global 0.0008 would flag half of them.
TINY_BOX_AREA_DEFAULT = 0.0008
TINY_BOX_AREA_BY_CLASS = {"pole": 0.0003, "woodenboard": 0.00015,
                          "squarepole": 0.0003, "hose": 0.00015}
EXTREME_ASPECT = 12.0      # w/h or h/w beyond this
JUMP_IOU = 0.10            # IoU with previous frame's box below this -> teleport
print(f"outputs root: {OUTPUTS_ROOT}")

## 2. Load all label sources

In [ ]:
BOX_COLS = ["sequence_id", "target_object", "camera", "frame_idx", "frame_path",
            "x0", "y0", "x1", "y1", "mask_area_frac"]

def load_boxes_tolerant(path: Path) -> pd.DataFrame:
    """Reads both current (x0..y1) and legacy (bbox_xyxy) row schemas."""
    import csv as _csv
    rows = []
    with path.open(newline="") as f:
        for rec in _csv.reader(f):
            if not rec or rec[0] == "sequence_id":
                continue
            if len(rec) == len(BOX_COLS):
                rows.append(dict(zip(BOX_COLS, rec)))
            elif len(rec) == 7:
                nums = re.findall(r"-?\d+", rec[5] or "")
                bbox = list(map(int, nums)) if len(nums) == 4 else [None] * 4
                rows.append({"sequence_id": rec[0], "target_object": rec[1],
                             "camera": rec[2], "frame_idx": rec[3], "frame_path": rec[4],
                             "x0": bbox[0], "y0": bbox[1], "x1": bbox[2], "y1": bbox[3],
                             "mask_area_frac": rec[6]})
    df = pd.DataFrame(rows, columns=BOX_COLS)
    for col in ["frame_idx", "x0", "y0", "x1", "y1", "mask_area_frac"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

frames_root_by_source: dict[str, Path] = {}
parts = []
for source in LABEL_SOURCES:
    if not Path(source["boxes_csv"]).exists():
        print(f"missing (skipped): {source['boxes_csv']}")
        continue
    df = load_boxes_tolerant(Path(source["boxes_csv"]))
    df["source"] = source["name"]
    frames_root_by_source[source["name"]] = Path(source["frames_root"])
    parts.append(df)
    print(f"{source['name']}: {len(df)} rows, {df['sequence_id'].nunique()} sequences")

labels = pd.concat(parts, ignore_index=True)
labels["has_box"] = labels["x0"].notna()

def resolve_frame(row) -> Path | None:
    p = Path(str(row["frame_path"]))
    if p.exists():
        return p
    alt = (frames_root_by_source[row["source"]]
           / str(row["sequence_id"]) / str(row["camera"]) / p.name)
    return alt if alt.exists() else None

## 3. Per-class statistics

In [ ]:
boxed = labels[labels["has_box"]].copy()
boxed["box_w"] = boxed["x1"] - boxed["x0"]
boxed["box_h"] = boxed["y1"] - boxed["y0"]
# frame size read lazily per sequence below; use mask_area_frac + pixel dims for stats
boxed["aspect"] = (boxed["box_w"] / boxed["box_h"].clip(lower=1)).clip(lower=1e-6)

stats = (
    labels.groupby(["target_object", "source"])
    .agg(sequences=("sequence_id", "nunique"),
         frames=("has_box", "size"),
         frames_with_box=("has_box", "sum"))
    .assign(coverage=lambda d: (d["frames_with_box"] / d["frames"]).round(3))
)
box_stats = (
    boxed.groupby(["target_object", "source"])
    .agg(area_frac_median=("mask_area_frac", "median"),
         box_w_median=("box_w", "median"),
         box_h_median=("box_h", "median"),
         aspect_median=("aspect", "median"))
    .round(4)
)
display(stats.join(box_stats))

## 4. Auto-flag suspicious labels

Flags are heuristics, not verdicts — they rank frames for human review:
`big_box` (tracker exploded onto background), `tiny_box` (collapsed mask),
`extreme_aspect`, `jump` (box teleported vs previous frame = drift/re-lock).

In [ ]:
def box_iou(a, b) -> float:
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1 - ix0) * max(0, iy1 - iy0)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-6)

flags = []
for (seq, cam), grp in boxed.sort_values("frame_idx").groupby(["sequence_id", "camera"]):
    prev_box = None
    for _, row in grp.iterrows():
        cur_box = (row["x0"], row["y0"], row["x1"], row["y1"])
        reasons = []
        if row["mask_area_frac"] > BIG_BOX_AREA:
            reasons.append("big_box")
        tiny_threshold = TINY_BOX_AREA_BY_CLASS.get(row["target_object"],
                                                    TINY_BOX_AREA_DEFAULT)
        if row["mask_area_frac"] < tiny_threshold:
            reasons.append("tiny_box")
        if row["aspect"] > EXTREME_ASPECT or row["aspect"] < 1 / EXTREME_ASPECT:
            reasons.append("extreme_aspect")
        if prev_box is not None and box_iou(prev_box, cur_box) < JUMP_IOU:
            reasons.append("jump")
        if reasons:
            flags.append({**row[["sequence_id", "target_object", "camera",
                                 "frame_idx", "frame_path", "source"]].to_dict(),
                          "x0": row["x0"], "y0": row["y0"],
                          "x1": row["x1"], "y1": row["y1"],
                          "flags": "|".join(reasons)})
        prev_box = cur_box

flags_df = pd.DataFrame(flags)
flags_df.to_csv(QA_DIR / "flagged_labels.csv", index=False)
print(f"flagged rows: {len(flags_df)} of {len(boxed)} "
      f"({100 * len(flags_df) / max(len(boxed), 1):.1f}%) -> {QA_DIR / 'flagged_labels.csv'}")
if len(flags_df):
    display(flags_df.groupby(["target_object", "flags"]).size().to_frame("n"))
    worst_sequences = (flags_df.groupby("sequence_id").size()
                       .sort_values(ascending=False).head(10))
    print("\nsequences with most flags (re-click candidates):")
    display(worst_sequences.to_frame("n_flags"))

## 5. Galleries: random sample per class (green) and flagged frames (red)

In [ ]:
def draw_box(img, row, color):
    x0, y0, x1, y1 = int(row["x0"]), int(row["y0"]), int(row["x1"]), int(row["y1"])
    cv2.rectangle(img, (x0, y0), (x1, y1), color, 3)
    cv2.putText(img, f"{row['target_object']} f{int(row['frame_idx'])}",
                (x0, max(y0 - 8, 16)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return img

def show_gallery(rows: pd.DataFrame, title: str, color=(0, 255, 0), cols=4):
    items = []
    for _, row in rows.iterrows():
        p = resolve_frame(row)
        if p is None:
            continue
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        items.append(draw_box(img, row, color))
    if not items:
        print(f"{title}: no renderable frames")
        return
    n = len(items)
    rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 3.4 * rows_n), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img in zip(axes.flat, items):
        ax.imshow(img)
    fig.suptitle(title, fontsize=13)
    fig.savefig(QA_DIR / f"gallery_{re.sub(r'[^a-z0-9]+', '_', title.lower())}.jpg",
                dpi=90, bbox_inches="tight")
    plt.show()

rng = random.Random(SEED)
for obj, grp in boxed.groupby("target_object"):
    sample_idx = rng.sample(range(len(grp)), min(N_GALLERY_PER_CLASS, len(grp)))
    show_gallery(grp.iloc[sample_idx], f"{obj} - random sample ({len(grp)} boxes total)")

if len(flags_df):
    for obj, grp in flags_df.groupby("target_object"):
        sample = grp.head(N_GALLERY_PER_CLASS)
        show_gallery(sample, f"{obj} - FLAGGED ({len(grp)} flagged)", color=(255, 40, 40))

## 6. Sequence filmstrips (drift check)

All frames of a sequence in one strip - drift shows up as the box sliding off the
object toward the end of the approach. Shown: the 3 most-flagged sequences plus one
random sequence per class.

In [ ]:
def show_filmstrip(seq_id: str, cam: str):
    grp = boxed[(boxed["sequence_id"] == seq_id)
                & (boxed["camera"] == cam)].sort_values("frame_idx")
    items = []
    for _, row in grp.iterrows():
        p = resolve_frame(row)
        if p is None:
            continue
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        img = draw_box(img, row, (0, 255, 0))
        img = cv2.resize(img, (320, 214))
        cv2.putText(img, f"f{int(row['frame_idx'])}", (6, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        items.append(img)
    if not items:
        return
    strip = np.hstack(items)
    fig, ax = plt.subplots(figsize=(min(2.2 * len(items), 40), 3))
    ax.imshow(strip)
    ax.axis("off")
    obj = grp["target_object"].iloc[0]
    ax.set_title(f"{seq_id} ({obj}, {cam})", fontsize=10)
    plt.show()

shown = set()
if len(flags_df):
    for seq_id in flags_df.groupby("sequence_id").size().sort_values(ascending=False).head(3).index:
        cam = flags_df[flags_df["sequence_id"] == seq_id]["camera"].iloc[0]
        show_filmstrip(seq_id, cam)
        shown.add(seq_id)

for obj, grp in boxed.groupby("target_object"):
    candidates = [s for s in grp["sequence_id"].unique() if s not in shown]
    if candidates:
        seq_id = rng.choice(sorted(candidates))
        cam = grp[grp["sequence_id"] == seq_id]["camera"].iloc[0]
        show_filmstrip(seq_id, cam)

## 7. Verdict + next steps

Save the class summary, then decide per class:

- **Clean** (few flags, galleries look right) -> ready for the training set.
- **Fixable** (specific sequences flagged) -> re-click those in
  `click_assisted_labeling.ipynb` (delete their `clicks.json` entries or use
  `REDO_OBJECTS`), re-propagate, rerun this notebook.
- **Systematic problem** (whole class drifts) -> change the prompt strategy for that
  class (e.g. box prompts, re-seeding mid-sequence) before training.

After QA passes here: build the batch SAM3 text-prompt labeler for the passing classes
(curbstone / speedbump / pole / ...), run THIS notebook on its output too, then merge
sources, split by sequence, and train.

In [ ]:
summary_out = QA_DIR / "qa_class_summary.csv"
stats.join(box_stats).to_csv(summary_out)
print(f"saved {summary_out}")
print(f"flagged labels : {QA_DIR / 'flagged_labels.csv'}")
print(f"galleries      : {QA_DIR}/gallery_*.jpg")